# DATN AI-SOAR — Huấn luyện dữ liệu thu bổ sung HTTP v1

Chỉ dùng dataset mới theo kế hoạch đã đóng băng trong G7 v3, được kiểm raw và phê duyệt **đúng manifest trong Git**. Candidate thu dữ liệu có trạng thái COLLECTION_READY chưa đủ để chạy train: sau khi thu và audit, Thịnh phải phát hành SHA review READY. Không dùng bốn file từ bộ 40 run cũ.

56 đặc trưng lưu trữ giữ nguyên 24 đặc trưng gốc và 20 đặc trưng stream. Mô hình Account dùng 10 đầu vào; C2 dùng 24 đầu vào đã khai báo. Đánh giá khả năng phân biệt **hành vi HTTP trong lab, trên ngày/máy mới của template đã biết**. Không suy luận ý đồ độc hại từ một lần đăng nhập, không tuyên bố bao phủ DNS-C2 hay C2 ngoài lab. Mọi package giữ ASSISTED/SHADOW, automation_eligible=false.

Chạy trên CPU. Bắt đầu phiên Colab mới, sao chép notebook vào Drive để giữ output, điền SHA được bàn giao rồi chạy lần lượt từ trên xuống. Không đổi threshold/gate để lấy PASS. Không dùng tập test khóa cho lựa chọn model.


## 0. Cấu hình phiên chạy

Trước khi chạy, tạo đúng thư mục Drive `MyDrive/DATN/supplemental-data/processed-supplemental-v1` và chỉ đặt bốn file an toàn cho Colab vào đó:

- `train.jsonl`
- `validation.jsonl`
- `dataset_manifest.json`
- `colab-upload.sha256`

Không tải dữ liệu raw, khóa/API key, `.env` hoặc tập test khóa lên Colab.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/lethinhlord/Repo.git"
BRANCH = "main"
# Transport branch only. Paste the future owner-approved READY review SHA after new data audit.
EXPECTED_TRAINING_COMMIT = ""
PROJECT_DIR = Path("/content/datn-ai-soar")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/DATN/supplemental-data/processed-supplemental-v1")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/DATN/supplemental-results")
EXPECTED_DATA_FILES = ("train.jsonl", "validation.jsonl", "dataset_manifest.json")
CHECKSUM_FILE = "colab-upload.sha256"
EXPECTED_FILES = (*EXPECTED_DATA_FILES, CHECKSUM_FILE)
RANDOM_SEED = 42

print(f"Repo: {REPO_URL}")
print(f"Branch: {BRANCH}")
print(f"Data source: {DRIVE_DATA_DIR}")
print(f"Result destination: {DRIVE_OUTPUT_DIR}")

## 1. Kiểm tra source và manifest

Điền EXPECTED_TRAINING_COMMIT bằng SHA 40 ký tự của review READY cho bộ mới, do Thịnh bàn giao sau audit. Nhánh main chỉ để tải lịch sử; checkout luôn detached đúng SHA. Cell kiểm hash bốn file, kế hoạch, source, schema và chứng nhận dataset trước khi cài thư viện. Nếu chưa có bộ mới được phê duyệt, dừng tại đây; không thay bằng SHA candidate thu dữ liệu.


In [ ]:
import hashlib
import importlib
import json
import os
import platform
import re
import subprocess
import sys

# Read only the staged manifest before obtaining or executing project code.
# Mount Drive first; a fresh Colab runtime must use a fresh checkout.
from google.colab import drive

drive.mount("/content/drive")
if not re.fullmatch(r"[0-9a-f]{40}", EXPECTED_TRAINING_COMMIT):
    raise RuntimeError(
        "Set EXPECTED_TRAINING_COMMIT to the owner-approved 40-character training SHA"
    )
if (DRIVE_DATA_DIR / "locked_test.jsonl").exists():
    raise RuntimeError("Locked test must never be staged on Drive")
if PROJECT_DIR.exists():
    raise RuntimeError("Existing project refused: start a fresh runtime/checkout")
missing = [name for name in EXPECTED_FILES if not (DRIVE_DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing staging files: {missing}")
checksum_rows = (DRIVE_DATA_DIR / CHECKSUM_FILE).read_text(encoding="utf-8").splitlines()
expected_hashes = {}
for line in checksum_rows:
    if not line.strip():
        continue
    digest, name = line.split()
    name = name.lstrip("*")
    if name in expected_hashes or not re.fullmatch(r"[0-9a-f]{64}", digest):
        raise RuntimeError("Invalid/duplicate staging checksum")
    expected_hashes[name] = digest
if set(expected_hashes) != set(EXPECTED_DATA_FILES):
    raise RuntimeError("Checksum must cover exactly the three permitted data files")
manifest_path = DRIVE_DATA_DIR / "dataset_manifest.json"
if hashlib.sha256(manifest_path.read_bytes()).hexdigest() != expected_hashes[manifest_path.name]:
    raise RuntimeError("SHA-256 mismatch: dataset_manifest.json")
staged_manifest = json.loads((DRIVE_DATA_DIR / "dataset_manifest.json").read_text(encoding="utf-8"))
provenance = staged_manifest.get("provenance", {})
builder_commit = provenance.get("builder_git_commit", "")
if not isinstance(builder_commit, str) or not re.fullmatch(r"[0-9a-f]{40}", builder_commit):
    raise RuntimeError("Invalid dataset builder identity")
if staged_manifest.get("manifest_version") != "dataset-manifest-v5-supplemental-v1":
    raise RuntimeError("This notebook requires the exact reviewed supplemental dataset")
if staged_manifest.get("locked_test_status") != "UNOPENED":
    raise RuntimeError("Locked test status must be UNOPENED")
# Equivalent command: git clone --branch main ...; branch is transport, SHA is authority.
subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "checkout", "--detach", EXPECTED_TRAINING_COMMIT],
    check=True,
)
os.chdir(PROJECT_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if commit != EXPECTED_TRAINING_COMMIT:
    raise RuntimeError("Checkout differs from approved source")
config_path = PROJECT_DIR / "ml/training_supplemental_config.yaml"
config_hash = hashlib.sha256(config_path.read_bytes()).hexdigest()
if config_hash != provenance.get("builder_config_sha256"):
    raise RuntimeError("Training config differs from dataset builder config")
print("Python:", platform.python_version())
design_path = PROJECT_DIR / "config/supplemental-design.yaml"
design_hash = hashlib.sha256(design_path.read_bytes()).hexdigest()
if (staged_manifest.get("design_sha256") != hashlib.sha256(
        json.dumps(staged_manifest["design"], sort_keys=True, separators=(",", ":")).encode()
    ).hexdigest() or staged_manifest.get("group_key") != "evaluation_group_id"
    or staged_manifest.get("evaluation_protocol") != "known-template-new-host-day"):
    raise RuntimeError("Registered design or dependency-group protocol differs")
print("Verified source:", commit)
print("Verified collection design:", design_hash)

# This stdlib-only verifier comes from the approved detached checkout, before pip.
# COLLECTION_READY cannot authorize training; verify_manifest requires the exact READY review.
sys.path.insert(0, str(PROJECT_DIR))

verify_training_manifest = importlib.import_module("ml.supplemental_sources").verify_manifest

verify_training_manifest(staged_manifest, root=PROJECT_DIR, expected_commit=commit)
for name in EXPECTED_DATA_FILES:
    if hashlib.sha256((DRIVE_DATA_DIR / name).read_bytes()).hexdigest() != expected_hashes[name]:
        raise RuntimeError(f"SHA-256 mismatch: {name}")
print("Verified training source compatibility")


## 2. Tạo môi trường huấn luyện Python 3.12

Kernel Colab gắn Drive và hiển thị kết quả. Việc kiểm dữ liệu, EDA, Ruff, toàn bộ pytest và huấn luyện chạy bằng cùng một Python 3.12 cô lập. Dùng các wheel đúng SHA-256 trong `requirements-runtime.lock` của API, sau đó thêm dependency cho kiểm thử và notebook với cùng các phiên bản lõi. Cell ghi phiên bản thực thi và SHA của lock vào log được xuất cùng package.

In [ ]:
import platform
import shutil

from ml.command_logging import run_logged

if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("This bootstrap requires the standard x86_64 Linux Colab CPU runtime")
EVIDENCE_DIR = PROJECT_DIR / "operator-logs" / "supplemental-colab"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
SESSION_LOG = EVIDENCE_DIR / "supplemental-colab-training.log"
TRAIN_ENV = PROJECT_DIR.parent / "supplemental-training-venv"
UV_TOOLS = PROJECT_DIR.parent / "supplemental-uv-bootstrap"
if TRAIN_ENV.exists() or UV_TOOLS.exists():
    raise RuntimeError("Existing training/tools environment: start a fresh Colab runtime")
TRAIN_PYTHON = TRAIN_ENV / "bin" / "python"
UV = UV_TOOLS / "bin" / "uv"

# Official PyPI uv 0.13.9 Linux x86_64 wheel. Keep the bootstrap out of the Colab kernel.
uv_spec = EVIDENCE_DIR / "uv-bootstrap.txt"
uv_spec.write_text(
    "uv==0.13.9 --hash=sha256:d7f459fe4fd247d830351c75686d3996ace3c6f3f728eb0e85a30b7ada0cef19\n",
    encoding="utf-8",
)
run_logged("Install verified uv bootstrap", [
    sys.executable, "-m", "pip", "--isolated", "install", "--index-url", "https://pypi.org/simple",
    "--only-binary=:all:", "--no-deps", "--require-hashes", "--target", UV_TOOLS, "-r", uv_spec,
], SESSION_LOG)
run_logged("Create Python 3.12 environment", [
    UV, "venv", "--managed-python", "--python", "3.12.15", "--seed", TRAIN_ENV,
], SESSION_LOG)
train_env = {
    **os.environ, "VIRTUAL_ENV": str(TRAIN_ENV),
    "PATH": str(TRAIN_PYTHON.parent) + os.pathsep + os.environ["PATH"],
    "PYTHONNOUSERSITE": "1",
}
train_env.pop("PYTHONPATH", None)
runtime_lock = PROJECT_DIR / "requirements-runtime.lock"
core_pins = [line.split()[0] for line in runtime_lock.read_text().splitlines()
             if re.match(r"^[A-Za-z0-9_.-]+==", line)]
constraints = EVIDENCE_DIR / "runtime-constraints.txt"
constraints.write_text("\n".join(core_pins) + "\n", encoding="utf-8")
install = [TRAIN_PYTHON, "-u", "-m", "pip", "--isolated", "install", "--index-url",
           "https://pypi.org/simple", "--only-binary=:all:"]
run_logged("Install hash-locked deployment packages", [
    *install, "--require-hashes", "-r", runtime_lock,
], SESSION_LOG, env=train_env)
run_logged("Install quality and notebook packages", [
    *install, "-c", constraints, "-r", "requirements.lock", "-r", "requirements-ml.lock",
    "-r", "requirements-notebook.lock",
], SESSION_LOG, env=train_env)
run_logged("Check dependency consistency", [
    TRAIN_PYTHON, "-u", "-m", "pip", "check",
], SESSION_LOG, env=train_env)
quality_passed = False


In [ ]:
verify_runtime = r"""
import hashlib
import json
import platform
import sys
from importlib.metadata import version
from pathlib import Path

if (sys.implementation.name != "cpython" or sys.version_info[:3] != (3, 12, 15)
        or sys.prefix == sys.base_prefix):
    raise RuntimeError("Training requires isolated CPython 3.12.15")
locked = {}
for pin in Path(sys.argv[1]).read_text().splitlines():
    name, expected = pin.split("==")
    locked[name] = version(name)
    if locked[name] != expected:
        raise RuntimeError(f"{name}={locked[name]} differs from deployment pin {expected}")
lock_hash = hashlib.sha256(Path("requirements-runtime.lock").read_bytes()).hexdigest()
print(json.dumps({
    "executable": sys.executable, "python": platform.python_version(),
    "runtime_lock_sha256": lock_hash,
    "runtime_packages": locked,
}, sort_keys=True))
"""
run_logged("Verify and record actual training runtime", [
    TRAIN_PYTHON, "-u", "-c", verify_runtime, constraints,
], SESSION_LOG, env=train_env)


## 3. Gắn Google Drive và nạp partition an toàn

Cell này chỉ sao chép ba file dữ liệu được phép và file checksum. Nếu nguồn hoặc project có `locked_test.jsonl`, phiên chạy dừng ngay.

In [ ]:
# Drive was mounted by the source gate before installing dependencies.


In [ ]:
import hashlib

SOURCE_LOCKED_TEST_PATH = DRIVE_DATA_DIR / "locked_test.jsonl"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed-supplemental-v1"
LOCKED_TEST_PATH = PROCESSED_DIR / "locked_test.jsonl"

if SOURCE_LOCKED_TEST_PATH.exists():
    raise RuntimeError(
        "Drive data folder contains locked_test.jsonl. Remove it from the Colab staging "
        "folder; do not inspect or upload the locked test during model selection."
    )

missing = [name for name in EXPECTED_FILES if not (DRIVE_DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"Thiếu {missing} trong {DRIVE_DATA_DIR}. Hãy build dataset trên Ubuntu trước."
    )

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
for name in EXPECTED_FILES:
    shutil.copy2(DRIVE_DATA_DIR / name, PROCESSED_DIR / name)

checksum_rows = (PROCESSED_DIR / CHECKSUM_FILE).read_text(encoding="utf-8").splitlines()
expected_hashes = {
    line.split()[1].lstrip("*"): line.split()[0]
    for line in checksum_rows
    if line.strip()
}
assert set(expected_hashes) == set(EXPECTED_DATA_FILES), expected_hashes
for name in EXPECTED_DATA_FILES:
    actual = hashlib.sha256((PROCESSED_DIR / name).read_bytes()).hexdigest()
    assert actual == expected_hashes[name], f"SHA-256 mismatch: {name}"

if LOCKED_TEST_PATH.exists():
    raise RuntimeError("Locked test boundary violated: refusing to continue.")

print("Staged files:")
for name in EXPECTED_FILES:
    path = PROCESSED_DIR / name
    print(f"- {path.name}: {path.stat().st_size:,} bytes")

## 4. Kiểm tra kế hoạch, các tập và leakage

Train có chứa thông tin trực tiếp hoặc gián tiếp về validation/test không? Cùng máy và ngày UTC phải ở cùng evaluation_group_id; các role đã được đăng ký trước thu. Số run/seed không phải số mẫu độc lập. Template có thể lặp giữa các role theo protocol known-template-new-host-day; không được gọi đây là đánh giá kỹ thuật tấn công chưa từng thấy.

Cell chỉ đọc TRAIN/VALIDATION và cam kết manifest; không mở tập test khóa. Schema, membership, vai trò, row hashes, context, raw provenance và source phải khớp review. EDA chỉ dùng TRAIN. Bất kỳ lỗi nào phải điều tra từ raw, không sửa nhãn/feature hoặc đổi split sau khi nhìn kết quả.


In [ ]:
preflight = r"""
import json
import sys
from collections import Counter
from pathlib import Path

PROJECT_DIR = Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data/processed-supplemental-v1"
commit = sys.argv[1]
output = Path(sys.argv[2])

from ml.dataset import load_feature_schema
from ml.training import exploratory_summary, leakage_audit, verify_training_partitions

train_rows, validation_rows, manifest = verify_training_partitions(
    PROCESSED_DIR, PROJECT_DIR / "ml" / "feature_schema_http.json",
    config_path=PROJECT_DIR / "ml/training_supplemental_config.yaml",
    expected_commit=commit,
    require_collection_audit=True,
    collection_design_path=PROJECT_DIR / "config/supplemental-design.yaml",
)
schema = load_feature_schema(PROJECT_DIR / "ml" / "feature_schema_http.json")
audit = leakage_audit(train_rows, validation_rows)

assert manifest.get("locked_test_status") == "UNOPENED", manifest
assert manifest.get("group_key") == "evaluation_group_id"
assert manifest.get("feature_schema_version") == "features-v5-http-v1"
assert manifest.get("collection_audit", {}).get("status") == "SUPPLEMENTAL_COLLECTION_AUDIT_PASS"
assert audit["passed"], audit
assert not ({r["run_id"] for r in train_rows} & {r["run_id"] for r in validation_rows})
assert not ({r["evaluation_group_id"] for r in train_rows}
            & {r["evaluation_group_id"] for r in validation_rows})
assert all(r["purpose"] == "trainable" for r in train_rows + validation_rows)
for partition_name, rows in (("train", train_rows), ("validation", validation_rows)):
    strata = Counter((row["family"], row["label"]) for row in rows)
    required_strata = {("account", 0), ("account", 1), ("c2", 0), ("c2", 1)}
    missing_strata = required_strata - set(strata)
    assert not missing_strata, f"{partition_name} thiếu strata: {missing_strata}"
    print(partition_name, "records=", len(rows), "strata=", dict(sorted(strata.items())))

print("Dataset hash:", manifest["dataset_hash_sha256"])
print("Feature schema:", manifest["feature_schema_version"])
print("Locked test status:", manifest["locked_test_status"])
print("Leakage audit:", json.dumps(audit, indent=2))

eda = exploratory_summary(train_rows, schema)
eda["independent_host_days"] = {
    family: len({r["evaluation_group_id"] for r in train_rows if r["family"] == family})
    for family in ("account", "c2")
}
(output / "eda.json").write_text(json.dumps(eda), encoding="utf-8")
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

for family, feature_order in schema["x-family-features"].items():
    family_rows = [row for row in train_rows if row["family"] == family]
    columns = 3
    rows = (len(feature_order) + columns - 1) // columns
    figure, axes = plt.subplots(rows, columns, figsize=(15, 3.8 * rows))
    axes = np.atleast_1d(axes).ravel()
    for unused in axes[len(feature_order):]:
        unused.set_visible(False)
    for axis, feature in zip(axes[:len(feature_order)], feature_order, strict=True):
        for label, color, name in ((0, "#4C78A8", "benign"), (1, "#E45756", "malicious")):
            values = [row["features"][feature] for row in family_rows if row["label"] == label]
            axis.hist(
                values,
                bins=min(15, max(3, len(set(values)))),
                alpha=0.65,
                label=name,
                color=color,
            )
        axis.set_title(f"{family}: {feature}")
        axis.set_xlabel(feature)
        axis.set_ylabel("count")
        axis.legend()
    plt.tight_layout()
    figure.savefig(output / f"eda-{family}.png")
    plt.close(figure)
"""
run_logged("Verify visible partitions and generate TRAIN EDA", [
    TRAIN_PYTHON, "-u", "-c", preflight, commit, EVIDENCE_DIR,
], SESSION_LOG, env=train_env)
manifest = json.loads((PROCESSED_DIR / "dataset_manifest.json").read_bytes())


## 5. EDA chỉ trên TRAIN

EDA chỉ dùng TRAIN để xem cân bằng nhãn, missing, khoảng giá trị và outlier. Validation không được dùng để quyết định feature hay làm sạch dữ liệu. Với bài toán cảnh báo mất cân bằng, metric chính là PR-AUC/F1 thay vì accuracy.

In [ ]:
import pandas as pd
from IPython.display import Image, display

eda = json.loads((EVIDENCE_DIR / "eda.json").read_bytes())
summary_rows = []
for family, family_stats in eda["families"].items():
    summary_rows.append(
        {
            "family": family,
            "records": family_stats["records"],
            "runs": family_stats["independent_runs"],
            "independent_host_days": eda["independent_host_days"][family],
            "benign": family_stats["benign"],
            "malicious": family_stats["malicious"],
        }
    )
display(pd.DataFrame(summary_rows))

feature_rows = []
for family, family_stats in eda["families"].items():
    for feature, stats in family_stats["features"].items():
        feature_rows.append({"family": family, "feature": feature, **stats})
display(pd.DataFrame(feature_rows))

In [ ]:
for family in eda["families"]:
    display(Image(filename=str(EVIDENCE_DIR / f"eda-{family}.png")))


## 6. Chạy quality gate của repo

Cả hai lệnh chạy bằng Python 3.12 trong môi trường vừa kiểm tra. Pytest chạy toàn bộ suite và in tên từng test. Log chung ghi thời gian bắt đầu, kết quả từng pha và output của tiến trình; file đang chạy nằm ở `operator-logs/supplemental-colab/supplemental-colab-training.log`. Lệnh tương đương:

```bash
python -m ruff check .
python -m pytest -q
```

In [ ]:
quality_passed = False
if shutil.which("node") is None:
    run_logged("Refresh Node package index", ["apt-get", "update", "-qq"], SESSION_LOG)
    run_logged("Install Node for workflow regression tests", [
        "apt-get", "install", "-y", "--no-install-recommends", "nodejs",
    ], SESSION_LOG)
run_logged("Node version", ["node", "--version"], SESSION_LOG, env=train_env)
run_logged("Ruff", [TRAIN_PYTHON, "-u", "-m", "ruff", "check", "."], SESSION_LOG, env=train_env)
run_logged("Full pytest suite", [
    TRAIN_PYTHON, "-u", "-m", "pytest", "-o", "addopts=", "-v", "--durations=15",
], SESSION_LOG, env=train_env)
quality_passed = True


## 7. Train và chọn model bằng grouped CV trên TRAIN

Pipeline chạy tương đương lệnh chuẩn:

```bash
python -m ml.train --config ml/training_supplemental_config.yaml
```

Model được chọn theo PR-AUC, F1 và Brier từ grouped CV trên TRAIN. Calibration chỉ fit trên TRAIN; VALIDATION chọn threshold nhắm precision tối thiểu 0,90 và kiểm calibration/độ ổn định. Nếu thiếu scenario validation độc lập, CI để null và automation_eligible=false.

In [ ]:
if not quality_passed:
    raise RuntimeError("Run the full quality gate successfully before training")
run_logged("Train and assess models", [
    TRAIN_PYTHON, "-u", "-m", "ml.train", "--config", "ml/training_supplemental_config.yaml",
], SESSION_LOG, env=train_env)


## 8. Đọc kết quả và kiểm tra tiêu chí chấp nhận

Không tuyên bố model tốt chỉ vì train thành công. Hãy đọc confidence interval, FPR, Brier, latency và trạng thái `automation_eligible` của từng family.

In [ ]:
import hashlib

PACKAGE_DIR = PROJECT_DIR / "ml" / "model-package-supplemental-v1"
required_outputs = (
    "model.joblib",
    "metadata.json",
    "metrics_validation.json",
    "thresholds.json",
    "eda_train.json",
    "leakage_audit.json",
    "calibration_validation.json",
    "feature_profile.json",
    "feature_schema.json",
)
missing_outputs = [name for name in required_outputs if not (PACKAGE_DIR / name).is_file()]
assert not missing_outputs, f"Thiếu model outputs: {missing_outputs}"

metadata = json.loads((PACKAGE_DIR / "metadata.json").read_text(encoding="utf-8"))
metrics = json.loads((PACKAGE_DIR / "metrics_validation.json").read_text(encoding="utf-8"))
thresholds = json.loads((PACKAGE_DIR / "thresholds.json").read_text(encoding="utf-8"))
package_audit = json.loads((PACKAGE_DIR / "leakage_audit.json").read_text(encoding="utf-8"))

assert metadata["locked_test_status"] == "UNOPENED"
assert metadata["dataset_hash_sha256"] == manifest["dataset_hash_sha256"]
assert metadata["dataset_provenance"] == manifest["provenance"]
assert metadata["dataset_manifest_payload_sha256"] == hashlib.sha256(
    json.dumps(manifest, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()
if manifest.get("manifest_version") == "dataset-manifest-v5-supplemental-v1":
    assert (metadata["dataset_source_compatibility"]
            == manifest["collection_audit"]["source_compatibility"])
assert metadata["evaluation_protocol"] == "known-template-new-host-day"
assert metadata["dependency_group_key"] == "evaluation_group_id"
assert metadata["supplemental_scope"]["automation_eligible"] is False
assert all(not x["automation_eligible"] for x in thresholds["families"].values())
assert package_audit["passed"]

candidate_rows = []
for family, family_result in metrics["families"].items():
    selected = family_result["selected_model"]
    for model_name, result in family_result["candidates"].items():
        values = result["inner_cv_metrics"]
        candidate_rows.append(
            {
                "family": family,
                "model": model_name,
                "selected": model_name == selected,
                "partition": "train_grouped_cv",
                "threshold": result["inner_cv_threshold"]["threshold"],
                "pr_auc": values["pr_auc"],
                "precision": values["precision"],
                "recall": values["recall"],
                "f1": values["f1"],
                "fpr": values["fpr"],
                "brier": values["brier"],
            }
        )

display(pd.DataFrame(candidate_rows).sort_values(["family", "selected"], ascending=[True, False]))
validation_rows_summary = []
for family, family_result in metrics["families"].items():
    selected = family_result["selected_model"]
    validation = family_result["candidates"][selected]["validation"]
    prior = family_result.get("prior_baseline", {}).get("metrics", {})
    validation_rows_summary.append({
        "family": family, "model": selected, "partition": "validation",
        "automation_eligible": thresholds["families"][family]["automation_eligible"],
        "pr_auc": validation["pr_auc"], "f1": validation["f1"], "fpr": validation["fpr"],
        "train_prior_pr_auc": prior.get("pr_auc"), "train_prior_f1": prior.get("f1"),
        "train_prior_fpr": prior.get("fpr"), "precision": validation["precision"],
        "recall": validation["recall"], "brier": validation["brier"], "ece": validation["ece"],
        "p95_ms": validation["latency"]["p95_per_record_ms"],
        "uncertainty_status": validation["uncertainty"]["status"],
    })
display(pd.DataFrame(validation_rows_summary))
print(json.dumps(thresholds, indent=2))
print("Model version:", metadata["model_version"])
print("Locked test status:", metadata["locked_test_status"])

## 9. Tạo báo cáo chạy và lưu model package vào Drive

Archive chứa model đã train, metadata, metric validation, EDA, leakage audit và log chung của môi trường Python, quality gate và quá trình train. Không chứa raw data hoặc locked test.

The package metadata and experiment summary retain the dataset provenance and reviewed source compatibility certificate. Their values must match the staged manifest before export.


In [ ]:
import hashlib
import shutil
import tarfile
from datetime import UTC, datetime

experiment_summary = {
    "experiment": "supplemental-http-v1",
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "git_commit": commit,
    "dataset_hash_sha256": metadata["dataset_hash_sha256"],
    "dataset_provenance": metadata["dataset_provenance"],
    "dataset_source_compatibility": metadata.get("dataset_source_compatibility"),
    "dataset_manifest_payload_sha256": metadata["dataset_manifest_payload_sha256"],
    "evaluation_protocol": metadata["evaluation_protocol"],
    "scope": metadata["supplemental_scope"],
    "split": f"grouped by {manifest['group_key']}; train/validation only",
    "random_seed": RANDOM_SEED,
    "locked_test_status": metadata["locked_test_status"],
    "selected_models": {
        family: family_result["selected_model"]
        for family, family_result in metrics["families"].items()
    },
    "automation_eligible": {
        family: result["automation_eligible"]
        for family, result in thresholds["families"].items()
    },
    "leakage_audit_passed": package_audit["passed"],
}
summary_path = EVIDENCE_DIR / "supplemental-colab-experiment-summary.json"
summary_path.write_text(json.dumps(experiment_summary, indent=2) + "\n", encoding="utf-8")

archive_path = PROJECT_DIR.parent / f"model-package-supplemental-{metadata['model_version']}.tar.gz"
with tarfile.open(archive_path, "x:gz") as archive:
    archive.add(PACKAGE_DIR, arcname="ml/model-package")
    archive.add(
        EVIDENCE_DIR / "supplemental-colab-training.log",
        arcname="evidence/supplemental-colab-training.log",
    )
    archive.add(summary_path, arcname="evidence/supplemental-colab-experiment-summary.json")

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
destination = DRIVE_OUTPUT_DIR / archive_path.name
# Exclusive destination preserves prior experiments. Publish success only after
# the complete copied bytes match the trusted archive, including a Drive readback.
with archive_path.open("rb") as source, destination.open("xb") as target:
    shutil.copyfileobj(source, target)
copied_hash = hashlib.sha256(destination.read_bytes()).digest()
if copied_hash != hashlib.sha256(archive_path.read_bytes()).digest():
    raise RuntimeError("Drive export hash mismatch; preserve failed output and investigate")
print("Saved:", destination)
print("Size:", f"{destination.stat().st_size / 1024 / 1024:.2f} MiB")
print(json.dumps(experiment_summary, indent=2))
print("Archive SHA-256:", hashlib.sha256(destination.read_bytes()).hexdigest())

## 10. Kết thúc và bước tiếp theo

Chỉ coi phiên train hoàn thành khi:

1. Ruff và toàn bộ pytest đều PASS.
2. Leakage audit có `passed: true`.
3. Cả `account` và `c2` đều có kết quả cho ba candidate.
4. Metadata ghi đúng dataset hash và `locked_test_status: UNOPENED`.
5. Archive xuất hiện trong `MyDrive/DATN/supplemental-results`.

Bàn giao package thật theo [Week 3 runbook](../docs/week3-model-training-runbook.md), mục 6, trên Ubuntu VM tin cậy. Chỉ dùng archive do phiên train tin cậy này tạo; không load joblib từ nguồn không tin cậy. Ghi lại `model_version`, dataset hash, commit và archive SHA-256 đã in ở trên qua kênh tin cậy. Đối chiếu SHA-256 file tải về với giá trị đó trước khi giải nén. `scripts/verify_week3_model.sh` chỉ tạo và kiểm tra package smoke riêng; smoke PASS không nghiệm thu package Colab.

Trên Ubuntu, ở thư mục gốc repo, đặt các biến dưới đây từ bằng chứng tin cậy của phiên Colab: `ARCHIVE` là đường dẫn tuyệt đối file đã tải, `EXPECTED_ARCHIVE_SHA256`, `EXPECTED_MODEL_VERSION`, `EXPECTED_GIT_COMMIT`, `EXPECTED_DATASET_SHA256`, và `TRUSTED_MANIFEST` là đường dẫn tuyệt đối manifest gốc. Xuất biến qua environment an toàn; không nhập khóa ký vào notebook. Lệnh sau là một khối duy nhất fail-stop: nếu đích đã tồn tại hoặc bất kỳ kiểm tra nào thất bại thì dừng trước freeze. Thư mục đích mới có thể còn trống hoặc dở dang sau lỗi; điều tra trước khi thử lại.

```bash
unset MODEL_PACKAGE_HOST_DIR MODEL_EVALUATION_REGISTRY_HOST_DIR
(
set -eu
: "${ARCHIVE:?}" "${EXPECTED_ARCHIVE_SHA256:?}" "${EXPECTED_MODEL_VERSION:?}"
: "${EXPECTED_GIT_COMMIT:?}" "${EXPECTED_DATASET_SHA256:?}" "${TRUSTED_MANIFEST:?}"
.venv-ml/bin/python -c 'import os,re; assert re.fullmatch(r"week3-[0-9]{8}T[0-9]{6}Z", os.environ["EXPECTED_MODEL_VERSION"])'
mkdir -p "$(pwd -P)/artifacts/runtime/model-releases"
MODEL_PACKAGE_DIR="$(pwd -P)/artifacts/runtime/model-releases/$EXPECTED_MODEL_VERSION"
mkdir "$MODEL_PACKAGE_DIR"
STAGE="$(mktemp -d)"
printf '%s  %s\n' "$EXPECTED_ARCHIVE_SHA256" "$ARCHIVE" | sha256sum -c -
.venv-ml/bin/python - "$ARCHIVE" "$STAGE" <<'PY'
import pathlib, sys, tarfile
archive, stage = sys.argv[1:]
seen = set()
with tarfile.open(archive, "r:gz") as tar:
    members = tar.getmembers()
    for member in members:
        name = member.name
        path = pathlib.PurePosixPath(name)
        allowed = (name == "ml/model-package" or name.startswith("ml/model-package/")
                   or name in {"evidence/supplemental-colab-training.log",
                               "evidence/supplemental-colab-experiment-summary.json"})
        if (not allowed or path.is_absolute() or ".." in path.parts or "\\" in name
                or member.issym() or member.islnk()
                or not (member.isfile() or member.isdir()) or name in seen):
            raise SystemExit("Unsafe or unexpected archive member: " + name)
        seen.add(name)
    tar.extractall(path=stage, members=members)
PY
.venv-ml/bin/python - "$STAGE" "$TRUSTED_MANIFEST" <<'PY'
import hashlib, json, os, pathlib, sys
stage = pathlib.Path(sys.argv[1])
package = stage / "ml/model-package"
metadata = json.loads((package / "metadata.json").read_text())
manifest = json.loads(pathlib.Path(sys.argv[2]).read_text())
from ml.supplemental_sources import verify_manifest
verify_manifest(manifest, root=pathlib.Path.cwd(), expected_commit=os.environ["EXPECTED_GIT_COMMIT"])
summary = json.loads((stage / "evidence/supplemental-colab-experiment-summary.json").read_text())
audit = json.loads((package / "leakage_audit.json").read_text())
checks = [
    metadata.get("model_version") == os.environ["EXPECTED_MODEL_VERSION"],
    metadata.get("git_commit") == os.environ["EXPECTED_GIT_COMMIT"],
    metadata.get("dataset_hash_sha256") == os.environ["EXPECTED_DATASET_SHA256"],
    metadata.get("dataset_hash_sha256") == manifest.get("dataset_hash_sha256"),
    metadata.get("dataset_provenance") == manifest.get("provenance"),
    manifest.get("manifest_version") == "dataset-manifest-v5-supplemental-v1",
    metadata.get("dataset_manifest_payload_sha256") == hashlib.sha256(
        json.dumps(manifest, sort_keys=True, separators=(",", ":")).encode()).hexdigest(),
    metadata.get("dataset_source_compatibility") == manifest.get("collection_audit", {}).get("source_compatibility"),
    summary.get("dataset_provenance") == metadata.get("dataset_provenance"),
    summary.get("dataset_source_compatibility") == metadata.get("dataset_source_compatibility"),
    summary.get("dataset_manifest_payload_sha256") == metadata.get("dataset_manifest_payload_sha256"),
    summary.get("git_commit") == metadata.get("git_commit"),
    summary.get("dataset_hash_sha256") == metadata.get("dataset_hash_sha256"),
    metadata.get("locked_test_status") == summary.get("locked_test_status") == "UNOPENED",
    audit.get("passed") is True,
]
if not all(checks):
    raise SystemExit("Package identity, provenance or leakage audit mismatch")
PY
cp -a "$STAGE/ml/model-package/." "$MODEL_PACKAGE_DIR/"
DATN_VERIFIED_PACKAGE_DIR="$MODEL_PACKAGE_DIR"
readonly DATN_VERIFIED_PACKAGE_DIR
set -a
. ./.env
set +a
MODEL_PACKAGE_DIR="$DATN_VERIFIED_PACKAGE_DIR"
export MODEL_EVALUATION_REGISTRY_DIR="$(pwd -P)/artifacts/runtime/model-releases/.model-evaluation-registry"
sudo --preserve-env=MODEL_PACKAGE_SIGNING_KEY,MODEL_PACKAGE_VERIFICATION_KEYS,MODEL_EVALUATION_REGISTRY_DIR \
  .venv-ml/bin/python -m ml.freeze_model --package-dir "$MODEL_PACKAGE_DIR"
sudo --preserve-env=MODEL_PACKAGE_SIGNING_KEY,MODEL_PACKAGE_VERIFICATION_KEYS,MODEL_EVALUATION_REGISTRY_DIR \
  .venv-ml/bin/python -m ml.freeze_model --verify-only --package-dir "$MODEL_PACKAGE_DIR"
sudo chmod -R a+rX,a-w "$MODEL_PACKAGE_DIR"
sudo chmod 700 "$MODEL_EVALUATION_REGISTRY_DIR"
printf 'Verified package: %s\n' "$MODEL_PACKAGE_DIR"
)
handoff_status=$?
if [ "$handoff_status" -eq 0 ]; then
    export MODEL_PACKAGE_HOST_DIR="$(pwd -P)/artifacts/runtime/model-releases/$EXPECTED_MODEL_VERSION"
    export MODEL_EVALUATION_REGISTRY_HOST_DIR="$(pwd -P)/artifacts/runtime/model-releases/.model-evaluation-registry"
else
    echo "Handoff failed; do not activate this package." >&2
    false
fi
```

Đọc validation metrics và các gate theo runbook trước activation. Đối chiếu `model_version` và đường dẫn package đã in với package được cấu hình vào image. Không dùng package khác nếu khối trên thất bại.
Sau khối thành công, `MODEL_PACKAGE_HOST_DIR` trỏ đến release host đã kiểm tra; đặt `MODEL_PACKAGE_DIR=/app/ml/model-package` cho container như runbook và giữ nguyên bind mount vào đường dẫn đó. Làm theo runbook để build và bật API SHADOW từ chính release này. Kiểm `/model/ready` báo `mode: trained`, `model_version` đúng package đã chọn, `locked_test_status: UNOPENED`; kiểm cả `/ready`. Nếu không khớp, dừng nghiệm thu. `automation_eligible=false` nghĩa là tiếp tục ASSISTED/SHADOW. Không mở locked test ở bước này.

Ubuntu handoff dùng UID 0 qua `sudo --preserve-env` với đúng ba biến signing/keyring/registry để khớp image API hiện hành (UID 0, `cap_drop: ALL`). Registry và signed head thuộc cùng UID; không thêm capability hoặc mở quyền world-writable. Colab chỉ tạo archive chưa freeze. Sau handoff, dùng cả `MODEL_PACKAGE_HOST_DIR` và `MODEL_EVALUATION_REGISTRY_HOST_DIR` trong mục 6 runbook: mục này chụp lại hai đường dẫn trước khi đọc `.env`, verify bằng cùng UID và lưu cả hai lựa chọn vào `.env` trước activation. Deployment dùng UID khác phải cấp ownership nhất quán từ freeze tới runtime.

**Profile HTTP:** trước khi dùng API, kiểm `/ready` trả `features-v5-http-v1`. Tạo incident bằng `/features/wazuh/http/batch` hoặc store stateful HTTP riêng, rồi gọi `/predict/http`. Workflow v3 hiện hữu vẫn giữ profile v3; không nối package 56 feature vào `/predict` hoặc cache v3. Chưa đổi n8n production trong bước huấn luyện này.

Không đổi n8n production hoặc bật phản ứng tự động sau phiên này. HTTP API và collector phải dùng cùng supplemental-sources.yaml; cache HTTP khóa theo cấu hình và baseline. Khi đổi cấu hình hoặc baseline, giữ cache cũ làm evidence và dùng đường dẫn cache mới. Số liệu validation tốt chỉ có ý nghĩa trong protocol đã đăng ký. Đánh giá locked test là thao tác riêng sau khi đóng băng model, không thuộc notebook này.
